# Interpretability & Code-Switching Analysis
Questo notebook analizza i casi di **Code-Switching (Italiano Standard ↔ Dialetti)** e le feature simboliche usate dal modello **Feature Fusion** nella classificazione.

## 1. Setup, Import e Caricamento dei Modelli Addestrati

In [ ]:
import sys
import os
sys.path.insert(0, "src")

import joblib
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer

from dataset import DIALECT_CLASSES, ID2LABEL
from models import BaselineBERT, DialectFeatureFusionBERT
from explainability import ModelPredictWrapper, inspect_auxiliary_features

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device in uso per l\u0027analisi: {device}")

model_name = "models/bert-base-italian-xxl-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

model_base = BaselineBERT(model_name=model_name, num_classes=len(DIALECT_CLASSES))
model_base.load_state_dict(torch.load("checkpoints/best_baseline_model.pt", map_location=device))
model_base.to(device).eval()

extractor = joblib.load("checkpoints/fusion_feature_extractor.joblib")
aux_dim = extractor.n_features_
model_fusion = DialectFeatureFusionBERT(model_name=model_name, num_classes=len(DIALECT_CLASSES), aux_dim=aux_dim)
model_fusion.load_state_dict(torch.load("checkpoints/best_fusion_model.pt", map_location=device))
model_fusion.to(device).eval()

print("-> Modelli ed estrattore caricati con successo!")

## 2. Selezione ed Esplorazione di Casi Reali di Code-Switching
Esempi di testi misti italiano-dialetto per confrontare le predizioni dei modelli.

In [ ]:
df_social = pd.read_csv('Social_media_posts/clean.csv')
print(f"Totale post social disponibili: {len(df_social):,}")

# Esempi selezionati ad alto tasso di code-switching (Campania/Napoli, Veneto, Sicilia, Lombardia)
sample_cases = [
    # Caso 1: Napoletano / Campano con matrice italiana
    "Andrea che non voleva alzarsi e fa teng na mutanda bianca",
    "Stamattina stavo andando a lavoro ma nun se po fa niente con sto traffico",
    # Caso 2: Veneto con espressioni e clitici
    "Tutti a parlare ma qua no se capise niente e no go mia tempo da perder",
    # Caso 3: Siciliano con shibboleth fonetico
    "Ma che cosa dici, chisto è veramente un beddu picciriddu",
    # Caso 4: Italiano standard di controllo
    "Oggi pomeriggio andiamo a fare una passeggiata in centro città con gli amici"
]

pred_wrapper_base = ModelPredictWrapper(model_base, tokenizer, device, is_fusion=False)
pred_wrapper_fusion = ModelPredictWrapper(model_fusion, tokenizer, device, is_fusion=True, extractor=extractor)


print("=== PREDIZIONI SUI CASI DI CODE-SWITCHING ===")
probs_base = pred_wrapper_base(sample_cases)
probs_fusion = pred_wrapper_fusion(sample_cases)

def entropy(p, eps=1e-12):
    return -(p * np.log(p + eps)).sum(axis=-1)

H_base = entropy(probs_base)       # shape (N,)
H_fusion = entropy(probs_fusion)

for i, text in enumerate(sample_cases):
    pred_b = ID2LABEL[np.argmax(probs_base[i])]
    conf_b = np.max(probs_base[i]) * 100
    pred_f = ID2LABEL[np.argmax(probs_fusion[i])]
    conf_f = np.max(probs_fusion[i]) * 100
    print(f'\n[{i+1}] Testo: "{text}"')
    print(f"    -> Baseline BERT:       {pred_b} ({conf_b:.1f}%)")
    print(f"    -> Feature Fusion BERT: {pred_f} ({conf_f:.1f}%)")
    print(f"    -> Entropia Baseline:       {H_base[i]:.3f}")
    print(f"    -> Entropia Feature Fusion: {H_fusion[i]:.3f}")

## 3. Ispezione Interna del Ramo Ausiliario (Feature Fusion)
Per comprendere perché il modello Feature Fusion prende una specifica decisione in presenza di code-switching, ispezioniamo quali delle **938 feature simboliche** si sono attivate con il massimo impatto proiettato.